# Demo guiada: los servicios Big Data locales y su símil en la nube

**Curso BIY7131 – DUOC 2026-1**

Este notebook recorre los **4 servicios** del entorno local (Docker) y los
conecta con su equivalente gestionado en la nube. La idea es que veas
*funcionando* cada pieza y entiendas qué problema resuelve.

| Pieza local (Docker) | ¿Qué hace? | Azure | AWS | Google Cloud |
|----------------------|-----------|-------|-----|--------------|
| **Spark** (en Jupyter) | Motor de cómputo distribuido | Synapse / HDInsight | EMR / Glue | Dataproc |
| **Kafka** | Bus de mensajería en tiempo real | Event Hubs | MSK / Kinesis | Pub/Sub |
| **Spark Structured Streaming** | Procesamiento continuo de flujos | Stream Analytics | Kinesis Data Analytics | Dataflow |
| **Hive Metastore + HiveServer2** | Catálogo de datos + motor SQL | Synapse SQL / Purview | Glue Data Catalog + Athena | Dataproc Metastore + BigQuery |

> **Requisito:** entorno levantado con el perfil **completo**
> (`docker compose --profile completo up -d`).
>
> **Idea fuerza:** en local *tú* operas cada servicio (lo instalas, lo
> configuras, lo mantienes). En la nube esos mismos servicios son
> *gestionados*: pagas por uso y el proveedor se encarga de la operación.
> Los **conceptos** son idénticos; cambia **quién opera**.

---
## 1️⃣ Spark — el motor de cómputo

Spark reparte el trabajo de procesar datos grandes en varias tareas
paralelas. Aquí leemos un CSV de vuelos y hacemos una agregación.

☁️ **En la nube:** un clúster de Spark gestionado → **Dataproc** (GCP),
**EMR** (AWS) o **Synapse/HDInsight** (Azure).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Una sola SparkSession para toda la demo.
# - El paquete spark-sql-kafka habilita leer/escribir Kafka (sección 3).
spark = (
    SparkSession.builder
    .appName("demo_servicios_vs_nube")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2")
    .config("spark.sql.shuffle.partitions", "4")  # dataset chico: pocas particiones
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "listo. UI: http://localhost:4040")

Spark 4.1.2 listo. UI: http://localhost:4040


In [2]:
# Leer un dataset y hacer una agregación distribuida
vuelos = spark.read.csv(
    "/home/jovyan/datos/flights.csv", header=True, inferSchema=True, nullValue="NA"
)
print(f"Filas: {vuelos.count():,}  |  Columnas: {len(vuelos.columns)}")

(
    vuelos.groupBy("carrier")
    .agg(F.count("*").alias("n_vuelos"),
         F.round(F.avg("dep_delay"), 1).alias("atraso_prom_min"))
    .orderBy(F.desc("n_vuelos"))
    .show(10)
)

Filas: 162,049  |  Columnas: 16


+-------+--------+---------------+
|carrier|n_vuelos|atraso_prom_min|
+-------+--------+---------------+
|     AS|   62460|            2.8|
|     WN|   23355|           13.3|
|     OO|   18710|            4.4|
|     DL|   16716|            4.8|
|     UA|   16671|            9.8|
|     AA|    7586|           10.6|
|     US|    5946|            2.7|
|     B6|    3540|            8.5|
|     VX|    3272|            7.9|
|     F9|    2698|           10.2|
+-------+--------+---------------+
only showing top 10 rows


👆 Spark dividió el CSV en particiones y procesó la agregación en paralelo.
Abre **http://localhost:4040** para ver los *jobs* y *stages*: ése es el mismo
panel que verías (gestionado) en la consola de Dataproc/EMR.

---
## 2️⃣ Kafka — el bus de mensajería en tiempo real

Kafka transporta eventos entre quien los **produce** y quien los **consume**,
desacoplándolos. Vamos a producir transacciones simuladas hacia un *topic* y
luego consumirlas.

☁️ **En la nube:** **Pub/Sub** (GCP), **Kinesis / MSK** (AWS) o
**Event Hubs** (Azure).

In [3]:
import json
from kafka import KafkaProducer

TOPIC = "demo_clase_tx"

# Dentro de la red de Docker, el broker se llama "kafka:29092".
productor = KafkaProducer(
    bootstrap_servers="kafka:29092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

regiones = ["Norte", "Centro", "Sur"]
productos = ["Laptop", "Mouse", "Monitor", "Teclado", "Audífonos"]
import random
random.seed(7)

for i in range(1, 31):
    evento = {
        "id": f"tx_{i:04d}",
        "region": random.choice(regiones),
        "producto": random.choice(productos),
        "total": random.randint(8000, 1500000),
        "cantidad": random.randint(1, 5),
    }
    productor.send(TOPIC, value=evento)

productor.flush()
productor.close()
print(f"Enviados 30 eventos al topic '{TOPIC}'")

Enviados 30 eventos al topic 'demo_clase_tx'


In [4]:
from kafka import KafkaConsumer

# Consumir desde el inicio del topic (solo para la demo).
consumidor = KafkaConsumer(
    TOPIC,
    bootstrap_servers="kafka:29092",
    auto_offset_reset="earliest",
    consumer_timeout_ms=5000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

print("Primeros 5 eventos leídos del topic:\n")
for n, msg in enumerate(consumidor):
    print(f"  offset {msg.offset:>2} → {msg.value}")
    if n >= 4:
        break
consumidor.close()

Primeros 5 eventos leídos del topic:



  offset  0 → {'id': 'tx_0001', 'region': 'Centro', 'producto': 'Mouse', 'total': 836004, 'cantidad': 1}
  offset  1 → {'id': 'tx_0002', 'region': 'Sur', 'producto': 'Laptop', 'total': 774905, 'cantidad': 5}
  offset  2 → {'id': 'tx_0003', 'region': 'Sur', 'producto': 'Mouse', 'total': 86634, 'cantidad': 1}
  offset  3 → {'id': 'tx_0004', 'region': 'Centro', 'producto': 'Laptop', 'total': 512706, 'cantidad': 1}
  offset  4 → {'id': 'tx_0005', 'region': 'Norte', 'producto': 'Audífonos', 'total': 267631, 'cantidad': 2}


👆 El **productor** no sabe quién consume y el **consumidor** lee a su ritmo.
Ese desacople es la razón de existir de Kafka/Pub/Sub: muchos productores y
consumidores independientes sobre el mismo flujo de eventos.

---
## 3️⃣ Spark Structured Streaming — procesamiento continuo

Ahora **Spark lee el topic de Kafka como un flujo** y agrega en tiempo real.
Usamos el *trigger* `availableNow` para procesar todo lo disponible y detenernos
(ideal para una demo; en producción el flujo queda corriendo).

☁️ **En la nube:** **Dataflow** (GCP), **Kinesis Data Analytics** (AWS) o
**Stream Analytics** (Azure).

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Esquema de los eventos que pusimos en Kafka
esquema = StructType([
    StructField("id", StringType()),
    StructField("region", StringType()),
    StructField("producto", StringType()),
    StructField("total", LongType()),
    StructField("cantidad", LongType()),
])

# Leer Kafka como STREAM (no como tabla estática)
flujo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

# El value viene como bytes -> lo pasamos a JSON con el esquema
eventos = (
    flujo.selectExpr("CAST(value AS STRING) AS json")
    .select(F.from_json("json", esquema).alias("e"))
    .select("e.*")
)

# Agregación por región
agg = eventos.groupBy("region").agg(
    F.count("*").alias("n_tx"),
    F.sum("total").alias("monto_total"),
)

# Escribir el resultado a una tabla en memoria y detenerse al terminar
consulta = (
    agg.writeStream.format("memory")
    .queryName("ventas_por_region")
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
print("Streaming procesado. Resultado agregado:\n")
spark.sql("SELECT * FROM ventas_por_region ORDER BY monto_total DESC").show()

Streaming procesado. Resultado agregado:



+------+----+-----------+
|region|n_tx|monto_total|
+------+----+-----------+
|   Sur|  22|   17587128|
|Centro|  20|   14490090|
| Norte|  18|   12391580|
+------+----+-----------+



👆 El mismo código de DataFrame sirve para datos estáticos y para **flujos**:
ésa es la gracia de Structured Streaming. En la nube, Dataflow/Stream Analytics
hacen exactamente esto pero escalando solos y sin que administres servidores.

---
## 4️⃣ Hive — el catálogo de datos (metastore) y el motor SQL

El **Hive Metastore** es un *catálogo*: guarda qué tablas existen, sus columnas
y dónde viven los datos. **HiveServer2** es un motor SQL que consulta esas
tablas. Juntos permiten tratar archivos en disco como **tablas SQL**.

☁️ **En la nube:** el catálogo es **Glue Data Catalog** (AWS),
**Dataproc Metastore** (GCP) o **Purview/Synapse** (Azure); el motor SQL sobre
él es **Athena** (AWS) o **BigQuery** (GCP).

A diferencia de Spark o Kafka (que usamos desde Python), con Hive interactuamos
a través de su propio cliente SQL, **beeline**, que se conecta a HiveServer2.
Ejecuta esto **en una terminal** (no en el notebook) para ver el catálogo:

```bash
docker exec -it bigdata-hive-server \
  /opt/hive/bin/beeline -u "jdbc:hive2://localhost:10000/" \
  -e "SHOW DATABASES;"
```

Y para crear y consultar una tabla SQL real:

```bash
docker exec -it bigdata-hive-server \
  /opt/hive/bin/beeline -u "jdbc:hive2://localhost:10000/" -e "
    CREATE DATABASE IF NOT EXISTS clase;
    CREATE TABLE IF NOT EXISTS clase.ventas (region STRING, monto INT) STORED AS PARQUET;
    INSERT INTO clase.ventas VALUES ('Norte', 1000), ('Sur', 2500);
    SELECT region, SUM(monto) FROM clase.ventas GROUP BY region;
  "
```

> **Por qué por terminal y no desde Spark:** en este entorno el cliente Hive que
> trae Spark 4.1 (versión 2.3) no es compatible con el metastore Hive 4.0, por lo
> que las operaciones de tabla desde Spark fallan o son muy lentas. El motor
> nativo de Hive (HiveServer2 + beeline) es la forma correcta y fiable de
> trabajar el catálogo. En la nube este problema no existe: el catálogo y el
> motor SQL ya vienen integrados y versionados por el proveedor.

---
## Cierre — ¿quién opera qué?

| Pregunta | Local (este entorno) | Nube |
|----------|----------------------|------|
| ¿Quién instala y actualiza? | **Tú** (Docker) | El proveedor |
| ¿Quién escala el clúster? | **Tú** (recursos del PC) | Automático / a demanda |
| ¿Cómo se paga? | Hardware propio | Por uso (cómputo + almacenamiento) |
| ¿Dónde viven los datos? | Volúmenes Docker | Object storage (S3 / GCS / Blob) |
| Conceptos (topics, particiones, metastore, ventanas) | **Idénticos** | **Idénticos** |

La habilidad que te llevas es **entender los conceptos**: una vez que sabes qué
es un *topic*, una partición, un metastore o una ventana de tiempo, da igual si
el servicio se llama Kafka o Pub/Sub. Eso es lo que transfieres a la nube.

In [6]:
# Liberar recursos al terminar la demo
spark.stop()
print("Sesión Spark detenida. Fin de la demo.")

Sesión Spark detenida. Fin de la demo.
